# Notebook 1: Data Pipeline Siêu Tốc Độ (3.000 Bài Báo - Xoay Vòng 7 Key Gemini)
---
**Cấu hình hiện tại:**
- **7 API Keys từ 7 Gmail độc lập** đã được cài sẵn vào hệ thống.
- **Mục tiêu:** ~2.700 - 3.000 bài báo khoa học đỉnh cao (15 chuyên đề x 200 bài) cho cả 3 Agents (Scope, Criteria, Keywords).
- **Cơ chế chống chết:** Nếu 1 key hết hạn mức (Quota), code sẽ tự động nhảy sang key tiếp theo mà không làm gián đoạn quá trình.

In [ ]:
# Cell 1: Cài đặt thư viện
!pip install requests pandas google-generativeai tqdm scikit-learn

In [ ]:
# Cell 2: Khởi tạo 7 Gemini API Keys xoay vòng
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import google.generativeai as genai
import json
import time
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ==========================================================================
# 🔑 DANH SÁCH 7 API KEYS TỪ 7 GMAIL CỦA BẠN (ĐÃ NẠP SẴN)
# ==========================================================================
GEMINI_API_KEYS = [
    "YOUR_API_KEY_HERE",
    "YOUR_API_KEY_HERE",
    "YOUR_API_KEY_HERE",
    "YOUR_API_KEY_HERE",
    "YOUR_API_KEY_HERE",
    "YOUR_API_KEY_HERE",
    "YOUR_API_KEY_HERE",
]

ACTIVE_KEYS = list(GEMINI_API_KEYS)
print(f"🚀 ĐÃ NẠP THÀNH CÔNG {len(ACTIVE_KEYS)} API KEYS ĐỘC LẬP!")
print(f"⚡ TỔNG HẠN MỨC MIỄN PHÍ: ~{len(ACTIVE_KEYS) * 1500} REQUESTS (DƯ SỨC CHO BỘ DATA 3.000 BÀI)!")

# 15 chuyên đề x 200 bài = 3.000 BÀI BÁO
PAPERS_PER_SUBTOPIC = 200

## Bước 1: Quét Toàn Diện 15 Chuyên Đề 3 Ngành (Toán, Y tế, Robotics)

In [ ]:
# Cell 3: Cào 3.000 bài báo từ ArXiv (hoặc load lại nếu đã có file)
if os.path.exists('raw_abstracts_massive.csv'):
    raw_df = pd.read_csv('raw_abstracts_massive.csv')
    print(f'⚡ [ĐÃ CÓ SẴN FILE CACHE]: Đọc thành công {len(raw_df)} bài báo thô từ raw_abstracts_massive.csv!')
else:
    subtopics = [
        # TOÁN HỌC & TỐI ƯU HÓA
        ("optimization stochastic gradient descent SGD convergence", "Toán học & Tối ưu hóa"),
        ("physics informed neural networks PINNs", "Toán học & Tối ưu hóa"),
        ("machine learning theory generalization bounds", "Toán học & Tối ưu hóa"),
        ("convex optimization non-convex deep learning", "Toán học & Tối ưu hóa"),
        ("adaptive gradient methods Adam RMSprop theory", "Toán học & Tối ưu hóa"),
        
        # Y TẾ & CHẨN ĐOÁN Y SINH
        ("medical image segmentation MRI CT scan", "Y tế & Chẩn đoán Y sinh"),
        ("few-shot learning biomedical imaging", "Y tế & Chẩn đoán Y sinh"),
        ("ECG signal classification deep learning arrhythmia", "Y tế & Chẩn đoán Y sinh"),
        ("3D medical segmentation brain tumor", "Y tế & Chẩn đoán Y sinh"),
        ("histopathology cancer detection deep learning", "Y tế & Chẩn đoán Y sinh"),
        
        # ROBOTICS & HỆ THỐNG TỰ HÀNH
        ("deep reinforcement learning robotic manipulation", "Robotics & Hệ thống tự hành"),
        ("MuJoCo physics simulation robot control", "Robotics & Hệ thống tự hành"),
        ("Isaac Sim robot learning autonomous navigation", "Robotics & Hệ thống tự hành"),
        ("visual SLAM lidar autonomous vehicles", "Robotics & Hệ thống tự hành"),
        ("bipedal quadruped robot locomotion RL", "Robotics & Hệ thống tự hành"),
    ]

    def fetch_arxiv_bulk(query, domain_name, max_results):
        url = f'http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}'
        try:
            response = requests.get(url, timeout=20)
            root = ET.fromstring(response.content)
            abstracts = []
            for entry in root.findall('{http://www.w3.org/2005/Atom}entry'):
                title = entry.find('{http://www.w3.org/2005/Atom}title').text.strip().replace('\n', ' ')
                summary = entry.find('{http://www.w3.org/2005/Atom}summary').text.strip().replace('\n', ' ')
                abstracts.append({'title': title, 'abstract': summary, 'domain': domain_name})
            return abstracts
        except Exception as e:
            print(f'Lỗi tải query [{query}]: {e}')
            return []

    all_papers = []
    for query, domain in subtopics:
        print(f'-> Đang quét ({PAPERS_PER_SUBTOPIC} bài): {query[:35]}... ({domain})')
        papers = fetch_arxiv_bulk(query, domain, PAPERS_PER_SUBTOPIC)
        all_papers.extend(papers)
        time.sleep(0.5)

    raw_df = pd.DataFrame(all_papers).drop_duplicates(subset=['title'])
    raw_df.to_csv('raw_abstracts_massive.csv', index=False, encoding='utf-8-sig')
    print(f'\n💥 [TỔNG CỘNG ĐÃ THU ĐƯỢC]: {len(raw_df)} bài báo khoa học thô!')

## Bước 2: Prompt Templates (Sử dụng chuỗi raw / replace an toàn)

In [ ]:
# Cell 4: Templates dùng placeholder rõ ràng tránh xung đột JSON brace
agent1_prompt = """You are an expert AI professor in __DOMAIN__. Read this paper abstract:
"__ABSTRACT__"

Based on this, invent a naive research idea a student might propose that is "too_broad" or "too_narrow".
Then, act as the professor and evaluate it.
Output strictly valid JSON with this schema:
{
  "instruction": "Evaluate the research scope and suggest refinements.",
  "input": "Domain: __DOMAIN__\nTopic: __TITLE__\nIdea: [Naive student idea in Vietnamese]",
  "output": "{\"status\": \"[too_broad or too_narrow]\", \"feedback\": \"[Expert feedback in Vietnamese]\", \"suggested_topics\": [\"[Refined topic 1 in Vietnamese]\", \"[Refined topic 2 in Vietnamese]\"]}"
}
Ensure the "output" field is a stringified JSON. No markdown tags."""

agent2_prompt = """You are a strict methodology expert in __DOMAIN__. Read this paper abstract:
"__ABSTRACT__"

Write PRISMA inclusion and exclusion criteria for a systematic literature review on this topic.
Output strictly valid JSON with this schema:
{
  "instruction": "Generate rigorous PRISMA inclusion and exclusion criteria.",
  "input": "Domain: __DOMAIN__\nTopic: __TITLE__",
  "output": "{\"include\": [\"[Include criteria 1 in Vietnamese]\", \"[Include criteria 2 in Vietnamese]\"], \"exclude\": [\"[Exclude criteria 1 in Vietnamese]\", \"[Exclude criteria 2 in Vietnamese]\"]}"
}
Ensure the "output" field is a stringified JSON. No markdown tags."""

agent3_prompt = """You are a technical research librarian expert in __DOMAIN__. Read this paper abstract:
"__ABSTRACT__"

Extract the PICO elements and construct a Boolean academic search query string (AND, OR, parentheses).
Output strictly valid JSON with this schema:
{
  "instruction": "Extract PICO elements and generate a robust Boolean search query.",
  "input": "Topic: __TITLE__",
  "output": "{\"P\": \"[Problem/Population in English]\", \"I\": \"[Intervention/Method in English]\", \"C\": \"[Comparison/Baseline in English]\", \"O\": \"[Outcome/Metric in English]\", \"boolean_query\": \"[Robust boolean query string]\"}"
}
Ensure the "output" field is a stringified JSON. No markdown tags."""
print('-> Đã sẵn sàng Templates hoàn hảo!')

## Bước 3: Sinh Dataset 3.000 Mẫu Với 7 Key Tự Động Xoay Vòng & Auto-Save

In [ ]:
# Cell 5: Hàm gọi API xoay vòng 7 Key mượt mà và tự nhảy key khi gặp lỗi
key_index = 0
def get_next_model():
    global key_index
    current_key = ACTIVE_KEYS[key_index % len(ACTIVE_KEYS)]
    key_index += 1
    genai.configure(api_key=current_key)
    return genai.GenerativeModel('gemini-1.5-flash')

def generate_and_append_dataset(df, prompt_template, agent_name, output_filename):
    existing_count = 0
    if os.path.exists(output_filename):
        with open(output_filename, 'r', encoding='utf-8') as f:
            existing_count = sum(1 for _ in f)
            
    print(f'\n=== {agent_name.upper()} | Đã có: {existing_count} mẫu | Cần tạo thêm: {len(df) - existing_count} mẫu ===')
    
    with open(output_filename, 'a', encoding='utf-8') as f:
        for index, row in tqdm(df.iloc[existing_count:].iterrows(), total=len(df) - existing_count):
            # Thay thế an toàn bằng replace không lo lỗi KeyError của .format()
            prompt = (prompt_template
                      .replace('__DOMAIN__', str(row.get('domain', '')))
                      .replace('__ABSTRACT__', str(row.get('abstract', '')))
                      .replace('__TITLE__', str(row.get('title', ''))))
            
            # Thử tối đa 7 lần tương ứng 7 keys
            for retry in range(len(ACTIVE_KEYS)):
                try:
                    model = get_next_model()
                    response = model.generate_content(prompt)
                    clean_text = response.text.replace('```json', '').replace('```', '').strip()
                    data_point = json.loads(clean_text)
                    json.loads(data_point['output'])
                    
                    # Ghi ngay lập tức xuống đĩa cứng
                    f.write(json.dumps(data_point, ensure_ascii=False) + '\n')
                    f.flush()
                    time.sleep(0.4) # Delay siêu ngắn vì có tận 7 keys phân tải!
                    break
                except Exception as e:
                    time.sleep(1)
                    continue

# Chạy lần lượt ghi dữ liệu vào 3 file tổng khổng lồ
generate_and_append_dataset(raw_df, agent1_prompt, 'Agent 1 (Scope)', 'raw_agent1_all.jsonl')
generate_and_append_dataset(raw_df, agent2_prompt, 'Agent 2 (Criteria)', 'raw_agent2_all.jsonl')
generate_and_append_dataset(raw_df, agent3_prompt, 'Agent 3 (Keywords)', 'raw_agent3_all.jsonl')

## Bước 4: Tách Thành Train (85%) và Test (15%) Cho 3.000 Mẫu

In [ ]:
# Cell 6: Tách file tổng thành Train/Test
def split_final_file(raw_filename, prefix):
    if not os.path.exists(raw_filename):
        print(f'Chưa tìm thấy {raw_filename}')
        return
    with open(raw_filename, 'r', encoding='utf-8') as f:
        lines = [json.loads(line) for line in f if line.strip()]
        
    train, test = train_test_split(lines, test_size=0.15, random_state=42)
    
    with open(f'{prefix}_train.jsonl', 'w', encoding='utf-8') as f:
        for row in train:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            
    with open(f'{prefix}_test.jsonl', 'w', encoding='utf-8') as f:
        for row in test:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            
    print(f'🔥 [{prefix.upper()} HOÀN THÀNH]: Train ({len(train)} mẫu) | Test ({len(test)} mẫu)')

split_final_file('raw_agent1_all.jsonl', 'agent1_scope')
split_final_file('raw_agent2_all.jsonl', 'agent2_criteria')
split_final_file('raw_agent3_all.jsonl', 'agent3_pico')

print('\n🏆 TOÀN BỘ DATASET ĐÃ HOÀN TẤT VÀ CHIA TRAIN/TEST SẴN SÀNG!')